In [2]:
# Mount Drive and verify environment

from google.colab import drive
drive.mount('/content/drive')

import os, torch, numpy as np
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

STGCN_CKPT       = '/content/drive/MyDrive/HRC_Research/checkpoints/stgcn_hri30/best_stgcn_hri30.pt'
CTRGCN_CKPT      = '/content/drive/MyDrive/HRC_Research/checkpoints/ctrgcn_hri30/best_ctrgcn_hri30.pt'
STGCN_TEST_DATA  = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/stgcn_format/test_data.npy'
STGCN_TEST_LABEL = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/stgcn_format/test_label.pkl'
CTRGCN_TEST_NPZ  = '/content/drive/MyDrive/HRC_Research/datasets/HRI30/ctrgcn_format/HRI30_CS.npz'
RESULTS_DIR      = '/content/drive/MyDrive/HRC_Research/results/occlusion_benchmark'
FIGURES_DIR      = '/content/drive/MyDrive/HRC_Research/figures'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

all_ok = True
checks = [
    ('ST-GCN checkpoint',  STGCN_CKPT),
    ('CTR-GCN checkpoint', CTRGCN_CKPT),
    ('ST-GCN test data',   STGCN_TEST_DATA),
    ('ST-GCN test labels', STGCN_TEST_LABEL),
    ('CTR-GCN npz',        CTRGCN_TEST_NPZ),
]
print('\n--- File verification ---')
for name, path in checks:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f'  OK  {name}: {size_mb:.1f} MB')
    else:
        print(f'  MISSING  {name}: {path}')
        all_ok = False

if all_ok:
    print('\n[PASS] All files found.')
else:
    print('\n[STOP] Fix missing files before continuing.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

--- File verification ---
  OK  ST-GCN checkpoint: 11.9 MB
  OK  CTR-GCN checkpoint: 5.8 MB
  OK  ST-GCN test data: 25.2 MB
  OK  ST-GCN test labels: 0.0 MB
  OK  CTR-GCN npz: 126.2 MB

[PASS] All files found.


In [3]:
# Clone repos, apply patches, load both models

import sys
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

STGCN_DIR  = '/content/st-gcn'
CTRGCN_DIR = '/content/CTR-GCN'

if not os.path.exists(STGCN_DIR):
    os.system(f'git clone https://github.com/yysijie/st-gcn.git {STGCN_DIR}')
    print('ST-GCN cloned.')
else:
    print('ST-GCN already present.')

stgcn_io = Path(f'{STGCN_DIR}/torchlight/torchlight/io.py')
text = stgcn_io.read_text()
if 'weights_only=False' not in text:
    text = text.replace(
        'torch.load(weights_path)',
        'torch.load(weights_path, weights_only=False, map_location="cpu")'
    )
    stgcn_io.write_text(text)
    print('ST-GCN torch.load patch applied.')

os.system(f'pip install -e {STGCN_DIR}/torchlight -q')

if not os.path.exists(CTRGCN_DIR):
    os.system(f'git clone https://github.com/Uason-Chen/CTR-GCN.git {CTRGCN_DIR}')
    print('CTR-GCN cloned.')
else:
    print('CTR-GCN already present.')

os.system('pip install -q tensorboardX torchpack fvcore iopath yacs thop')
os.system(f'pip install -e {CTRGCN_DIR}/torchlight -q')

ctrgcn_util = Path(f'{CTRGCN_DIR}/torchlight/torchlight/util.py')
text = ctrgcn_util.read_text()
if 'PaviLogger = None' not in text:
    text = text.replace(
        'from torchpack.runner.hooks import PaviLogger',
        'try:\n    from torchpack.runner.hooks import PaviLogger\nexcept ImportError:\n    PaviLogger = None'
    )
    ctrgcn_util.write_text(text)
    print('CTR-GCN torchpack patch applied.')

sys.path.insert(0, STGCN_DIR)
from net.st_gcn import Model as STGCN_Model

stgcn_model = STGCN_Model(
    in_channels=3, num_class=30, dropout=0.5,
    edge_importance_weighting=True,
    graph_args={'layout': 'ntu-rgb+d', 'strategy': 'spatial'}
)
stgcn_ckpt = torch.load(STGCN_CKPT, map_location='cpu', weights_only=False)
stgcn_state = stgcn_ckpt['model_state_dict'] if isinstance(stgcn_ckpt, dict) and 'model_state_dict' in stgcn_ckpt else stgcn_ckpt
stgcn_state = {k[7:] if k.startswith('module.') else k: v for k, v in stgcn_state.items()}
stgcn_model.load_state_dict(stgcn_state, strict=True)
stgcn_model = stgcn_model.to(device).eval()
print('ST-GCN loaded (30 classes) ✓')

sys.path.insert(0, CTRGCN_DIR)
from model.ctrgcn import Model as CTRGCN_Model

ctrgcn_model = CTRGCN_Model(
    num_class=30, num_point=25, num_person=1,
    graph='graph.ntu_rgb_d.Graph',
    graph_args={'labeling_mode': 'spatial'}
)
ctrgcn_ckpt = torch.load(CTRGCN_CKPT, map_location='cpu', weights_only=False)
ctrgcn_state = ctrgcn_ckpt['model_state_dict'] if isinstance(ctrgcn_ckpt, dict) and 'model_state_dict' in ctrgcn_ckpt else ctrgcn_ckpt
ctrgcn_state = {k[7:] if k.startswith('module.') else k: v for k, v in ctrgcn_state.items()}
ctrgcn_model.load_state_dict(ctrgcn_state, strict=True)
ctrgcn_model = ctrgcn_model.to(device).eval()
print('CTR-GCN loaded (30 classes) ✓')

print('\n[PASS] Both models loaded.')

Using device: cuda
ST-GCN cloned.
ST-GCN torch.load patch applied.
CTR-GCN cloned.
CTR-GCN torchpack patch applied.
ST-GCN loaded (30 classes) ✓
CTR-GCN loaded (30 classes) ✓

[PASS] Both models loaded.


In [6]:
# Load HRI30 test data

import pickle

# ST-GCN format
stgcn_test_data = np.load(STGCN_TEST_DATA)
with open(STGCN_TEST_LABEL, 'rb') as f:
    _, y_test = pickle.load(f)
stgcn_test_labels = np.array(y_test)

print(f'ST-GCN test data shape:   {stgcn_test_data.shape}')
print(f'ST-GCN test labels shape: {stgcn_test_labels.shape}')
print(f'Label range: {stgcn_test_labels.min()} to {stgcn_test_labels.max()}')

# CTR-GCN format
npz = np.load(CTRGCN_TEST_NPZ)
ctrgcn_test_data   = npz['x_test']
ctrgcn_test_labels = npz['y_test']

print(f'\nCTR-GCN test data shape:  {ctrgcn_test_data.shape}')
print(f'CTR-GCN test labels shape:{ctrgcn_test_labels.shape}')

# Verify both test sets are consistent
assert stgcn_test_data.shape[0] == ctrgcn_test_data.shape[0], 'Sample count mismatch!'
assert (stgcn_test_labels == ctrgcn_test_labels).all(), 'Label mismatch between formats!'

# Shared labels (both formats are identical)
y_true = stgcn_test_labels
N_TEST = len(y_true)

print(f'\n[PASS] {N_TEST} test samples, 30 classes, labels verified identical.')

ST-GCN test data shape:   (588, 3, 150, 25, 1)
ST-GCN test labels shape: (588,)
Label range: 0 to 29

CTR-GCN test data shape:  (588, 3, 150, 25, 1)
CTR-GCN test labels shape:(588,)

[PASS] 588 test samples, 30 classes, labels verified identical.


In [7]:
# Define shared inference functions

import torch.nn.functional as F

def apply_occlusion(data_np, occlusion_rate, seed=None):
    """
    Zeros out occlusion_rate% of the 25 joints for all samples.
    data_np: numpy array (N, C, T, V, M)
    occlusion_rate: float 0.0 to 1.0
    seed: integer seed for reproducibility
    Returns: corrupted copy of data_np
    """
    if occlusion_rate == 0.0:
        return data_np.copy()
    if seed is not None:
        np.random.seed(seed)
    data_out = data_np.copy()
    num_joints = data_out.shape[3]  # V = 25
    num_to_zero = max(1, int(round(num_joints * occlusion_rate)))
    joints_to_zero = np.random.choice(num_joints, num_to_zero, replace=False)
    data_out[:, :, :, joints_to_zero, :] = 0.0
    return data_out


def run_inference_with_softmax(model, data_np, batch_size=64):
    """
    Runs model inference and returns argmax predictions + softmax probabilities.
    data_np: (N, C, T, V, M) numpy array
    Returns:
        preds:   (N,) numpy array of predicted class indices
        softmax: (N, 30) numpy array of softmax probabilities
    """
    model.eval()
    all_preds, all_softmax = [], []
    N = data_np.shape[0]
    with torch.no_grad():
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            batch = torch.tensor(data_np[start:end], dtype=torch.float32).to(device)
            logits = model(batch)
            probs  = F.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)
            all_preds.append(preds.cpu().numpy())
            all_softmax.append(probs.cpu().numpy())
    return np.concatenate(all_preds), np.concatenate(all_softmax)


def entropy_weighted_consensus(st_softmax, ctr_softmax):
    """
    Entropy-weighted consensus fusion.
    w = exp(-H) normalised per sample.
    Exactly matches consensus_layer.py from Phase 4.
    Returns:
        con_preds:   (N,) argmax of fused probabilities
        con_softmax: (N, 30) fused probability distribution
    """
    eps = 1e-9
    # Entropy of each model's output for each sample
    H_st  = -np.sum(st_softmax  * np.log(st_softmax  + eps), axis=1)
    H_ctr = -np.sum(ctr_softmax * np.log(ctr_softmax + eps), axis=1)
    # Weights: lower entropy = higher confidence = higher weight
    w_st  = np.exp(-H_st)
    w_ctr = np.exp(-H_ctr)
    w_sum = w_st + w_ctr + eps
    w_st  = (w_st  / w_sum)[:, None]
    # Weighted sum of softmax outputs
    con_softmax = w_st * st_softmax + w_ctr * ctr_softmax
    con_preds   = con_softmax.argmax(axis=1)
    return con_preds, con_softmax


print('[PASS] Inference helpers defined. Proceed to Cell 5.')

[PASS] Inference helpers defined. Proceed to Cell 5.


In [8]:
# Run clean (0% occlusion) inference and verify against locked values

TOLERANCE = 0.01
LOCKED = {
    'ST-GCN':    61.22,
    'CTR-GCN':   69.05,
    'Consensus': 70.75,
}

print('Running ST-GCN on clean data...')
st_preds_clean, st_soft_clean = run_inference_with_softmax(stgcn_model, stgcn_test_data)
st_acc_clean = 100.0 * (st_preds_clean == y_true).mean()
print(f'  ST-GCN Top-1: {st_acc_clean:.4f}%')

print('Running CTR-GCN on clean data...')
ctr_preds_clean, ctr_soft_clean = run_inference_with_softmax(ctrgcn_model, ctrgcn_test_data)
ctr_acc_clean = 100.0 * (ctr_preds_clean == y_true).mean()
print(f'  CTR-GCN Top-1: {ctr_acc_clean:.4f}%')

print('Computing consensus...')
con_preds_clean, con_soft_clean = entropy_weighted_consensus(st_soft_clean, ctr_soft_clean)
con_acc_clean = 100.0 * (con_preds_clean == y_true).mean()
print(f'  Consensus Top-1: {con_acc_clean:.4f}%')

computed = {'ST-GCN': st_acc_clean, 'CTR-GCN': ctr_acc_clean, 'Consensus': con_acc_clean}
print('\n--- Locked value verification ---')
all_pass = True
for name, locked_val in LOCKED.items():
    comp_val = computed[name]
    diff = abs(comp_val - locked_val)
    status = '[PASS]' if diff <= TOLERANCE else '[FAIL]'
    if status == '[FAIL]':
        all_pass = False
    print(f'  {status} {name}: computed={comp_val:.4f}%, locked={locked_val:.2f}%, diff={diff:.4f} pp')

if not all_pass:
    raise RuntimeError(
        'LOCKED VALUE MISMATCH. Do NOT proceed.\n'
        'Report the computed values above to Claude before continuing.'
    )

print('\n[PASS] LOCKED VALUES VERIFIED.')

Running ST-GCN on clean data...
  ST-GCN Top-1: 61.2245%
Running CTR-GCN on clean data...
  CTR-GCN Top-1: 69.0476%
Computing consensus...
  Consensus Top-1: 70.7483%

--- Locked value verification ---
  [PASS] ST-GCN: computed=61.2245%, locked=61.22%, diff=0.0045 pp
  [PASS] CTR-GCN: computed=69.0476%, locked=69.05%, diff=0.0024 pp
  [PASS] Consensus: computed=70.7483%, locked=70.75%, diff=0.0017 pp

[PASS] LOCKED VALUES VERIFIED.


In [9]:
# Visual 1: Multi-metric comparison table (clean condition only)

from sklearn.metrics import precision_score, recall_score, f1_score
import csv

def compute_metrics(y_true, y_pred, name):
    """Compute macro Precision, Recall, F1 for one model's predictions."""
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0) * 100
    rec  = recall_score(   y_true, y_pred, average='macro', zero_division=0) * 100
    f1   = f1_score(       y_true, y_pred, average='macro', zero_division=0) * 100
    acc  = 100.0 * (y_pred == y_true).mean()
    return {'Model': name, 'Top-1 Acc (%)': f'{acc:.2f}', 'Precision (%)': f'{prec:.2f}',
            'Recall (%)': f'{rec:.2f}', 'F1-Score (%)': f'{f1:.2f}'}

results = [
    compute_metrics(y_true, st_preds_clean,  'ST-GCN'),
    compute_metrics(y_true, ctr_preds_clean, 'CTR-GCN'),
    compute_metrics(y_true, con_preds_clean, 'Consensus (ours)'),
]

header = ['Model', 'Top-1 Acc (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)']
col_w  = [20, 14, 14, 12, 13]
divider = '+' + '+'.join('-' * w for w in col_w) + '+'

print('\nMulti-Metric Comparison — Clean Condition (0% Occlusion), Macro Average')
print(divider)
print('|' + '|'.join(h.center(w) for h, w in zip(header, col_w)) + '|')
print(divider)
for row in results:
    print('|' + '|'.join(str(row[h]).center(w) for h, w in zip(header, col_w)) + '|')
print(divider)

OUT_CSV = f'{RESULTS_DIR.replace("occlusion_benchmark", "")}multiclass_metrics_clean.csv'

import os
OUT_CSV = os.path.join('/content/drive/MyDrive/HRC_Research/results', 'multiclass_metrics_clean.csv')
with open(OUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=header)
    writer.writeheader()
    writer.writerows(results)

size_kb = os.path.getsize(OUT_CSV) / 1024
print(f'\nCSV saved: {OUT_CSV} ({size_kb:.1f} KB)')
print('[PASS] Visual 1 complete.')


Multi-Metric Comparison — Clean Condition (0% Occlusion), Macro Average
+--------------------+--------------+--------------+------------+-------------+
|       Model        |Top-1 Acc (%) |Precision (%) | Recall (%) | F1-Score (%)|
+--------------------+--------------+--------------+------------+-------------+
|       ST-GCN       |    61.22     |    63.89     |   61.25    |    60.61    |
|      CTR-GCN       |    69.05     |    69.76     |   69.08    |    68.91    |
|  Consensus (ours)  |    70.75     |    71.82     |   70.76    |    70.48    |
+--------------------+--------------+--------------+------------+-------------+

CSV saved: /content/drive/MyDrive/HRC_Research/results/multiclass_metrics_clean.csv (0.2 KB)
[PASS] Visual 1 complete.


In [10]:
# Run 20% occlusion inference (seed=42, first trial)

OCCL_RATE = 0.20
SEED_20   = 42

print(f'Applying {int(OCCL_RATE*100)}% occlusion with seed={SEED_20}...')
print(f'Joints zeroed: max(1, round(25 * {OCCL_RATE})) = {max(1, round(25 * OCCL_RATE))}')

# Apply occlusion
st_data_20  = apply_occlusion(stgcn_test_data,   OCCL_RATE, seed=SEED_20)
ctr_data_20 = apply_occlusion(ctrgcn_test_data,  OCCL_RATE, seed=SEED_20)

print('Running ST-GCN at 20% occlusion...')
st_preds_20,  st_soft_20  = run_inference_with_softmax(stgcn_model,  st_data_20)
st_acc_20 = 100.0 * (st_preds_20 == y_true).mean()
print(f'  ST-GCN Top-1 at 20%: {st_acc_20:.4f}%')

print('Running CTR-GCN at 20% occlusion...')
ctr_preds_20, ctr_soft_20 = run_inference_with_softmax(ctrgcn_model, ctr_data_20)
ctr_acc_20 = 100.0 * (ctr_preds_20 == y_true).mean()
print(f'  CTR-GCN Top-1 at 20%: {ctr_acc_20:.4f}%')

print('Computing consensus at 20% occlusion...')
con_preds_20, _ = entropy_weighted_consensus(st_soft_20, ctr_soft_20)
con_acc_20 = 100.0 * (con_preds_20 == y_true).mean()
print(f'  Consensus Top-1 at 20%: {con_acc_20:.4f}%')

# Sanity check: this single trial result should be close to the locked mean (46.29%)
print(f'\n  [INFO] Locked mean for consensus at 20% = 46.29% (5-trial average)')
print(f'  [INFO] This single trial (seed=42) may differ; that is expected.')
print('\n[PASS] 20% occlusion inference complete.')

Applying 20% occlusion with seed=42...
Joints zeroed: max(1, round(25 * 0.2)) = 5
Running ST-GCN at 20% occlusion...
  ST-GCN Top-1 at 20%: 39.6259%
Running CTR-GCN at 20% occlusion...
  CTR-GCN Top-1 at 20%: 32.8231%
Computing consensus at 20% occlusion...
  Consensus Top-1 at 20%: 43.0272%

  [INFO] Locked mean for consensus at 20% = 46.29% (5-trial average)
  [INFO] This single trial (seed=42) may differ; that is expected.

[PASS] 20% occlusion inference complete.


In [13]:
# Visual 2: Confusion matrix heatmap (Consensus 0% vs 20%)

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix
import pandas as pd
import os

matplotlib.use('Agg')

cm_clean = confusion_matrix(y_true, con_preds_clean, labels=np.arange(30))
cm_20    = confusion_matrix(y_true, con_preds_20,    labels=np.arange(30))

def normalise_cm(cm):
    row_sums = cm.sum(axis=1, keepdims=True).astype(float)
    row_sums[row_sums == 0] = 1
    return (cm / row_sums) * 100.0

cm_clean_norm = normalise_cm(cm_clean)
cm_20_norm    = normalise_cm(cm_20)

def make_annot(cm_norm):
    df = pd.DataFrame(cm_norm)
    return df.applymap(lambda x: f'{x:.0f}' if x >= 5 else '')

annot_clean = make_annot(cm_clean_norm)
annot_20    = make_annot(cm_20_norm)

SHOW_TICKS  = [0, 5, 10, 15, 20, 25, 29]
tick_labels = [str(i) if i in SHOW_TICKS else '' for i in range(30)]

plt.rcParams.update({
    'font.family':  'serif',
    'font.serif':   ['Times New Roman', 'Times', 'DejaVu Serif'],
    'font.size':    9,
    'savefig.dpi':  300,
    'savefig.bbox': 'tight',
})


fig, axes = plt.subplots(1, 2, figsize=(10, 5))

configs = [
    (axes[0], cm_clean_norm, annot_clean, 'Consensus — 0% Occlusion (Clean)', False),
    (axes[1], cm_20_norm,    annot_20,    'Consensus — 20% Occlusion',         True),
]

for ax, cm_norm, annot_arr, title, show_cbar in configs:
    sns.heatmap(
        cm_norm,
        ax=ax,
        cmap='Blues',
        annot=annot_arr,
        fmt='',
        annot_kws={'size': 7},
        linewidths=0.3,
        linecolor='white',
        cbar=show_cbar,
        xticklabels=tick_labels,
        yticklabels=tick_labels,
        vmin=0, vmax=100
    )

    ax.set_title(title, fontsize=10, fontweight='bold', pad=8)
    ax.set_xlabel('Predicted Class', fontsize=9)
    ax.set_ylabel('True Class', fontsize=9)
    ax.tick_params(axis='both', labelsize=8)
plt.tight_layout(pad=1.5)

plt.show()
print('Figure displayed above. Confirm it looks correct.')
print('Run the save cell below only after confirming.')

/tmp/ipykernel_415/701304132.py:36: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: f'{x:.0f}' if x >= 5 else '')


Figure displayed above. Confirm it looks correct.
Run the save cell below only after confirming.


In [15]:
# Save confirmed figure to Drive

FIGURES_DIR = '/content/drive/MyDrive/HRC_Research/figures'
OUT_FIG = f'{FIGURES_DIR}/confusion_matrix_0_vs_20.png'

fig.savefig(OUT_FIG, dpi=300, bbox_inches='tight')
size_kb = os.path.getsize(OUT_FIG) / 1024
print(f'Saved: {OUT_FIG} ({size_kb:.1f} KB)')

Saved: /content/drive/MyDrive/HRC_Research/figures/confusion_matrix_0_vs_20.png (245.4 KB)


In [12]:
# Final output confirmation

outputs = [
    '/content/drive/MyDrive/HRC_Research/results/multiclass_metrics_clean.csv',
    '/content/drive/MyDrive/HRC_Research/figures/confusion_matrix_0_vs_20.png',
]

print('Output verification:')
all_ok = True
for path in outputs:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'  OK  {os.path.basename(path)}: {size_kb:.1f} KB')
    else:
        print(f'  MISSING  {path}')
        all_ok = False

if all_ok:
    print('\n[PASS] All outputs saved to Drive. Notebook complete.')
else:
    print('\n[STOP] Some outputs are missing. Check earlier cells.')

Output verification:
  OK  multiclass_metrics_clean.csv: 0.2 KB
  OK  confusion_matrix_0_vs_20.png: 232.0 KB

[PASS] All outputs saved to Drive. Notebook complete.
